# LFM Semantic Segmentation Comparison Workflow

## Purpose of this notebook
This notebook runs the existing toy DINO semantic segmentation model on the full-model split directory layout used by the Graha/Lunar-FM workflow. It uses the same helper functions as the sbatch script so notebook and batch behavior stay aligned while preserving the old model's baseline behavior.

## Imports, Dino Repo Clone

In [ ]:
import gc
import sys
from pathlib import Path

import torch

NOTEBOOK_DIR = Path.cwd().resolve()
LFM_ROOT = NOTEBOOK_DIR.parents[1]

if str(LFM_ROOT) not in sys.path:
    sys.path.insert(0, str(LFM_ROOT))

print("Notebook directory:", NOTEBOOK_DIR)
print("LFM root:", LFM_ROOT)

In [ ]:
from argparse import Namespace

from IPython.display import Image, display
from lightning.pytorch import seed_everything

from lfm.full_model import lfm_seg_finetuning_direct as graha_workflow
from lfm.full_model.utils import (
    create_timestamped_output_dir,
    plot_prediction_cache_comparison,
    save_prediction_cache,
)
from lfm.full_model.utils.utils import ensure_data_symlink
from toy_sem_seg_comparison import (
    build_config,
    create_datamodule,
    create_lightning_module,
    create_model,
    create_trainer,
    save_config,
    validate_data_paths,
)

## User Config

#### Paths
`INPUT_ROOT_DIR`: optional source directory containing the split-folder dataset. If set, the notebook creates `./data` as a symlink to this directory.

`DATA_ROOT`: optional explicit data directory. Leave as `None` to use `./data`, matching the sbatch script default.

`OUTPUT_DIR`: output root for checkpoints, config snapshots, file lists, logs, and visualizations.

`DINO_CHECKPOINT`: optional local DinoV3 checkpoint path. Leave as `None` to use the default path in `sseg_model.py`.

#### Dataset parameters
`BAND_FILTER`: list of input bands to keep, in order.

`TARGET_SIZE` and `SPATIAL_TRANSFORM`: control whether the split data is resized or cropped before entering the toy model. The comparison baseline uses a 256x256 center crop to better match the Graha/full-model path.

`MAX_*_SAMPLES`: optional per-split sample limits for smoke tests.

#### Training hyperparameters
`BATCH_SIZE`, `NUM_EPOCHS`, `BASE_LR`, and `WEIGHT_DECAY` control the training run.

#### Model hyperparameters
`FREEZE_ENCODER`: whether to keep the DinoV3 encoder frozen.

`LOSS_TYPE`: Lightning loss wrapper option for the toy comparison run.

#### Visualization
`PLOT_EVERY_N_EPOCHS` and `PLOT_N_SAMPLES` control validation prediction plots saved under the timestamped output directory.

#### Comparison cache
`CACHE_PREDICTIONS` saves lightweight prediction files for side-by-side comparison. The notebook runs and caches both the toy model and Graha/full model sequentially, then creates side-by-side plots from the saved caches.

In [ ]:
# Data paths
INPUT_ROOT_DIR = None  # Example: Path("/explore/nobackup/projects/lfm/model_inputs/my_sem_seg_split")
DATA_ROOT = None  # Leave as None to use NOTEBOOK_DIR / "data"
OUTPUT_DIR = "./outputs/toy_sem_seg_comparison"
DINO_CHECKPOINT = None  # Leave as None to use the default checkpoint in sseg_model.py

# Symlink setup
SIMLINK_DEST = INPUT_ROOT_DIR

# Dataset parameters
BAND_FILTER = [0, 1, 2, 3, 4, 5, 6]
TARGET_SIZE = 256
SPATIAL_TRANSFORM = "crop"  # "crop" or "resize"
MAX_TRAIN_SAMPLES = None
MAX_VAL_SAMPLES = None
MAX_TEST_SAMPLES = None

# Training hyperparameters
BATCH_SIZE = 16
NUM_WORKERS = 10
NUM_EPOCHS = 1  # Use 1 for smoke tests; set to 100 for comparison runs.
BASE_LR = 5e-5
WEIGHT_DECAY = 1e-3
LOSS_TYPE = "focal_dice"

# Model parameters
FREEZE_ENCODER = False

# Runtime controls
PLOT_EVERY_N_EPOCHS = 1
PLOT_N_SAMPLES = 5
CACHE_PREDICTIONS = True
PREDICTION_SPLIT = "val"
PREDICTION_N_SAMPLES = 20
SEED = 42
NO_FIT = False

# Graha/full-model comparison settings
GRAHA_PRETRAIN_DIR = None
GRAHA_OUTPUT_DIR = "./outputs/graha_finetuning"
GRAHA_CROP_SIZE = 256
GRAHA_STATS_BATCH_SIZE = 16
GRAHA_BATCH_SIZE = 16
GRAHA_NUM_WORKERS = 10
GRAHA_NUM_EPOCHS = NUM_EPOCHS
GRAHA_NO_FIT = NO_FIT

In [ ]:
DATA_SYMLINK = ensure_data_symlink(SIMLINK_DEST, NOTEBOOK_DIR / "data")

In [ ]:
args = Namespace(
    data_root=DATA_ROOT,
    base_output_dir=OUTPUT_DIR,
    dino_checkpoint=DINO_CHECKPOINT,
    band_filter=BAND_FILTER,
    target_size=TARGET_SIZE,
    spatial_transform=SPATIAL_TRANSFORM,
    max_train_samples=MAX_TRAIN_SAMPLES,
    max_val_samples=MAX_VAL_SAMPLES,
    max_test_samples=MAX_TEST_SAMPLES,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    max_epochs=NUM_EPOCHS,
    learning_rate=BASE_LR,
    weight_decay=WEIGHT_DECAY,
    loss_type=LOSS_TYPE,
    freeze_encoder=FREEZE_ENCODER,
    plot_every_n_epochs=PLOT_EVERY_N_EPOCHS,
    plot_n_samples=PLOT_N_SAMPLES,
    cache_predictions=CACHE_PREDICTIONS,
    prediction_split=PREDICTION_SPLIT,
    prediction_n_samples=PREDICTION_N_SAMPLES,
    seed=SEED,
    no_fit=NO_FIT,
)

config = build_config(args)
validate_data_paths(config)

print("Data root:", config.data_root)
print("Base output dir:", config.base_output_dir)
print("Band filter:", config.band_filter)
print("Target size:", config.target_size)
print("Spatial transform:", config.spatial_transform)
print("Max train/val/test samples:", config.max_train_samples, config.max_val_samples, config.max_test_samples)
print("Normalize inputs:", config.normalize_inputs)
print("Loss type:", config.loss_type)
print("Plot every N epochs:", config.plot_every_n_epochs)
print("Plot N samples:", config.plot_n_samples)
print("Cache predictions:", config.cache_predictions)
print("Prediction split/samples:", config.prediction_split, config.prediction_n_samples)
print("Max epochs:", config.max_epochs)

## Create output directory

In [ ]:
output_dir = create_timestamped_output_dir(config.base_output_dir)
save_config(config, output_dir)
print("Output dir:", output_dir)

## Create dataloaders

In [ ]:
seed_everything(config.seed)
datamodule = create_datamodule(config, output_dir)

if datamodule.weight_assignments is None:
    raise RuntimeError("DataModule did not create weight assignments.")

print("Weight assignments:", datamodule.weight_assignments)

## Load Encoder and Create Model

In [ ]:
model = create_model(config, datamodule.weight_assignments)
task = create_lightning_module(config, model)

print(type(model))
print(type(task))

## Trainer

In [ ]:
trainer = create_trainer(config, output_dir)

## Run Training

In [ ]:
# Set args.max_epochs = 1 in the config cell for a smoke test.
if args.no_fit:
    print("Skipping trainer.fit() because args.no_fit is True.")
else:
    trainer.fit(task, datamodule=datamodule)

## Test Best Checkpoint

In [ ]:
if args.no_fit:
    print("Skipping trainer.test() because args.no_fit is True.")
else:
    trainer.test(task, datamodule=datamodule, ckpt_path="best")

## Prediction cache and side-by-side comparison

The notebook can save lightweight prediction caches so different model runs can be compared without loading both models at once.

## Cache predictions for side-by-side comparison

In [ ]:
toy_prediction_cache = save_prediction_cache(
    task=task,
    datamodule=datamodule,
    output_dir=output_dir,
    model_name="toy",
    split=config.prediction_split,
    n_samples=config.prediction_n_samples,
)

## Release toy model memory

In [ ]:
del trainer, task, model, datamodule
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
print("Released toy model objects and cleared CUDA cache.")

## Run Graha/full model and cache predictions

In [ ]:
graha_workflow.configure_proj_environment()

graha_args = Namespace(
    data_root=DATA_ROOT,
    base_output_dir=GRAHA_OUTPUT_DIR,
    pretrain_dir=GRAHA_PRETRAIN_DIR,
    crop_size=GRAHA_CROP_SIZE,
    stats_batch_size=GRAHA_STATS_BATCH_SIZE,
    batch_size=GRAHA_BATCH_SIZE,
    num_workers=GRAHA_NUM_WORKERS,
    max_epochs=GRAHA_NUM_EPOCHS,
    cache_predictions=True,
    prediction_split=PREDICTION_SPLIT,
    prediction_n_samples=PREDICTION_N_SAMPLES,
    seed=SEED,
    no_fit=GRAHA_NO_FIT,
)

graha_config = graha_workflow.build_config(graha_args)
graha_workflow.configure_python_paths(graha_config)
graha_workflow.print_config(graha_config)
graha_workflow.validate_required_paths(graha_config)

In [ ]:
graha_deps = graha_workflow.import_project_dependencies()
graha_datamodule_cls = graha_deps["LunarSemanticSegmentationDatamodule"]
graha_task_cls = graha_workflow.make_notebook_task_class(
    graha_deps["LunarShapeSegmentationTask"]
)

graha_output_dir = graha_workflow.create_output_dirs(
    graha_config,
    graha_deps["create_timestamped_output_dir"],
)
seed_everything(graha_config.seed)

graha_means, graha_stds = graha_workflow.calculate_train_stats(
    graha_config,
    graha_datamodule_cls,
)
graha_datamodule = graha_workflow.create_datamodule(
    graha_config,
    graha_datamodule_cls,
    graha_means,
    graha_stds,
)
graha_sample_batch = graha_workflow.inspect_batch(graha_datamodule)
graha_task = graha_workflow.create_task(
    graha_config,
    graha_task_cls,
    graha_sample_batch,
)
graha_workflow.inspect_backbone(graha_task)
graha_trainer = graha_workflow.create_trainer(
    graha_config,
    graha_output_dir,
    graha_deps["ValidationPlotCallback"],
)

In [ ]:
if graha_args.no_fit:
    print("Skipping Graha trainer.fit() because GRAHA_NO_FIT is True.")
else:
    graha_trainer.fit(graha_task, datamodule=graha_datamodule)

graha_prediction_cache = save_prediction_cache(
    task=graha_task,
    datamodule=graha_datamodule,
    output_dir=graha_output_dir,
    model_name="graha",
    split=PREDICTION_SPLIT,
    n_samples=PREDICTION_N_SAMPLES,
)

## Release Graha/full model memory

In [ ]:
del graha_trainer, graha_task, graha_datamodule, graha_sample_batch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
print("Released Graha model objects and cleared CUDA cache.")

## Side-by-side comparison from cached predictions

This cell compares the toy and Graha/full-model prediction caches generated above.

In [ ]:
comparison_caches = {}
comparison_caches["toy"] = toy_prediction_cache
comparison_caches["graha"] = graha_prediction_cache

side_by_side_path = plot_prediction_cache_comparison(
    comparison_caches,
    output_dir / "comparison_plots",
    n_samples=5,
)
display(Image(filename=str(side_by_side_path)))

## Display validation visualizations

The training run saves validation prediction plots into the timestamped output directory.

In [ ]:
plot_dir = output_dir / "plots"
plot_files = sorted(plot_dir.glob("*.png")) if plot_dir.exists() else []

print(f"Found {len(plot_files)} validation plot(s) in {plot_dir}")
for path in plot_files:
    print(path)

In [ ]:
if plot_files:
    display(Image(filename=str(plot_files[-1])))
else:
    print("No validation plots found. Check PLOT_EVERY_N_EPOCHS and confirm training completed validation.")